<a href="https://colab.research.google.com/github/handdung/bigdata/blob/main/week04_5756498_%EC%9D%B4%ED%99%98%EC%A4%80.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 4주차 과제
빅데이터프로그래밍 · 통계학과

**이름:**
**학번:**

## 제출 방법
모든 셀을 실행해 출력이 보이는 상태로 저장한 뒤 `week04_학번_이름.ipynb` 로 제출합니다.

GPU 문제는 `런타임 > 런타임 유형 변경 > T4 GPU` 로 바꾼 뒤 실행하세요.


In [ ]:
import torch
import pandas as pd

device = "cuda" if torch.cuda.is_available() else "cpu"
print("torch", torch.__version__, "/ device:", device)

import time
import math

torch 2.11.0+cu130 / device: cuda


## 문제 1. Tensor 5개 생성과 속성 확인표 (40점)
서로 다른 Tensor 5개를 만들고, 아래 항목을 담은 표를 출력하세요.

| 이름 | shape | dim | dtype | device |

각 Tensor를 왜 그렇게 만들었는지 주석으로 한 줄 적으세요.


In [ ]:
# 답안
# 1) 스칼라: 값 하나짜리 0차원 텐서는 shape과 dim이 어떻게 나오는지 확인하려고
t_scalar = torch.tensor(3.14)

# 2) 벡터: 정수 리스트로 만들면 dtype이 자동으로 int64가 되는지 확인하려고
t_vector = torch.tensor([1, 2, 3, 4, 5])

# 3) 행렬: 표준정규 난수로 '관측치 3개 × 변수 4개' 같은 2차원 데이터를 표현하려고
t_matrix = torch.randn(3, 4)

# 4) 3차원: 이미지 한 장(채널×높이×너비)을 흉내 내고, float16으로 dtype 차이를 보려고
t_image = torch.zeros(3, 28, 28, dtype=torch.float16)

# 5) 4차원 배치: 이미지 8장 묶음을 GPU에 올려 device 차이를 확인하려고 (GPU 없으면 CPU)
t_batch = torch.ones(8, 1, 28, 28, device=device)

tensors = {
    't_scalar': t_scalar,
    't_vector': t_vector,
    't_matrix': t_matrix,
    't_image' : t_image,
    't_batch' : t_batch,
}

table = pd.DataFrame([
    {'이름': name,
     'shape': tuple(t.shape),
     'dim': t.dim(),
     'dtype': str(t.dtype),
     'device': str(t.device)}
    for name, t in tensors.items()
])
table

,이름,shape,dim,dtype,device
0,t_scalar,(),0,torch.float32,cpu
1,t_vector,"(5,)",1,torch.int64,cpu
2,t_matrix,"(3, 4)",2,torch.float32,cpu
3,t_image,"(3, 28, 28)",3,torch.float16,cpu
4,t_batch,"(8, 1, 28, 28)",4,torch.float32,cpu


## 문제 2. Tensor 연산 (20점)
(3, 4) 와 (4, 2) Tensor를 만들어 아래를 출력하세요.
- 행렬곱 결과와 그 shape
- 첫 번째 Tensor의 열별 평균과 행별 합계


In [ ]:
# 답안
# 결과를 손으로도 검산할 수 있도록 0~11, 0~7 값으로 생성
A = torch.arange(12, dtype=torch.float32).reshape(3, 4)   # (3, 4)
B = torch.arange(8,  dtype=torch.float32).reshape(4, 2)   # (4, 2)
print('A =\n', A)
print('B =\n', B)

# 행렬곱: (3,4) @ (4,2) -> (3,2)
C = A @ B            # torch.matmul(A, B) 와 동일
print('\n[행렬곱 결과]\n', C)
print('행렬곱 shape:', C.shape)

# 첫 번째 Tensor(A)의 열별 평균: dim=0 → 열마다 하나씩
print('\n[A의 열별 평균]', A.mean(dim=0))

# 첫 번째 Tensor(A)의 행별 합계: dim=1 → 행마다 하나씩
print('[A의 행별 합계]', A.sum(dim=1))

A =
 tensor([[ 0.,  1.,  2.,  3.],
        [ 4.,  5.,  6.,  7.],
        [ 8.,  9., 10., 11.]])
B =
 tensor([[0., 1.],
        [2., 3.],
        [4., 5.],
        [6., 7.]])

[행렬곱 결과]
 tensor([[ 28.,  34.],
        [ 76.,  98.],
        [124., 162.]])
행렬곱 shape: torch.Size([3, 2])

[A의 열별 평균] tensor([4., 5., 6., 7.])
[A의 행별 합계] tensor([ 6., 22., 38.])


## 문제 3. CPU와 GPU 비교 (20점)
같은 크기의 행렬곱을 CPU와 GPU에서 각각 실행해 걸린 시간을 출력하고, 차이를 한두 줄로 설명하세요.
GPU를 쓸 수 없는 경우에는 CPU 결과와 그 이유를 적으세요.


In [ ]:
# 답안
import time

N = 4000   # 4000 x 4000 행렬끼리 곱하기

x_cpu = torch.randn(N, N)
y_cpu = torch.randn(N, N)

# --- CPU ---
start = time.perf_counter()
z_cpu = x_cpu @ y_cpu
cpu_time = time.perf_counter() - start
print(f'CPU 시간: {cpu_time:.4f} 초')

CPU 시간: 0.9330 초


In [ ]:
# --- GPU ---
if torch.cuda.is_available():
    x_gpu = x_cpu.to('cuda')
    y_gpu = y_cpu.to('cuda')

    # 워밍업: 첫 실행은 CUDA 초기화 비용이 섞이므로 한 번 미리 돌림
    _ = x_gpu @ y_gpu
    torch.cuda.synchronize()

    start = time.perf_counter()
    z_gpu = x_gpu @ y_gpu
    torch.cuda.synchronize()   # GPU 연산은 비동기라서 끝날 때까지 기다려야 정확한 시간
    gpu_time = time.perf_counter() - start

    print(f'GPU 시간: {gpu_time:.4f} 초')
    print(f'GPU가 약 {cpu_time / gpu_time:.1f}배 빠름')
else:
    print('GPU를 사용할 수 없어 CPU 결과만 출력합니다.')
    print('이유: 현재 런타임에 CUDA GPU가 할당되지 않음 '
          '(Colab 런타임 유형이 CPU이거나 GPU 사용량 제한에 걸린 경우).')

GPU 시간: 0.0358 초
GPU가 약 26.1배 빠름


## 문제 4. 자동미분 (20점)
아래 두 가지를 하세요.
1. $y = 2x^3 + 5x$ 에서 $x=3$ 일 때의 도함수를 자동미분으로 구하고, 손으로 구한 값과 비교하세요.
2. 자기가 만든 수식 하나에 대해 미분값을 구하고, 손 계산과 일치하는지 확인하세요.


In [ ]:
# 답안
# 4-1.
x = torch.tensor(3.0, requires_grad=True)
y = 2 * x**3 + 5 * x
y.backward()                      # dy/dx 계산

auto_grad = x.grad.item()
hand_grad = 6 * 3**2 + 5          # 손으로 구한 값: 59

print('y 값        :', y.item())
print('자동미분 값 :', auto_grad)
print('손 계산 값  :', hand_grad)
print('일치 여부   :', math.isclose(auto_grad, hand_grad))

x = torch.tensor(math.pi / 2, dtype=torch.float64, requires_grad=True)
f = x**2 * torch.sin(x)
f.backward()

auto_grad = x.grad.item()
hand_grad = math.pi               # 손으로 구한 값

print('자동미분 값 :', auto_grad)
print('손 계산 값  :', hand_grad)
print('일치 여부   :', math.isclose(auto_grad, hand_grad, rel_tol=1e-9))

y 값        : 69.0
자동미분 값 : 59.0
손 계산 값  : 59
일치 여부   : True
자동미분 값 : 3.141592653589793
손 계산 값  : 3.141592653589793
일치 여부   : True
